# MedMon Kaggle Notebook
This notebook contains the complete code for the MedMon scraper, capable of running headlessly on Kaggle.

### Instructions:
1. **Run Cell 1** to install dependencies.
2. **Run Cell 2** to setup database functions.
3. **Run Cell 3** to setup scraper functions.
4. **Run Cell 4** to configure and START scraping.

In [ ]:
!pip install gnews newspaper3k googlenewsdecoder selenium transformers torch wordcloud matplotlib pandas lxml[html_clean]
!apt-get update
!apt-get install -y google-chrome-stable

In [ ]:
"""
Database operations for MedMon using SQLite (Kaggle Compatible)
"""
import sqlite3
from datetime import datetime
from contextlib import contextmanager
import os

DATABASE_NAME = 'medmon.db'

@contextmanager
def get_connection():
    """Context manager for database connections."""
    # check_same_thread=False is needed for Streamlit's threading model with SQLite
    conn = sqlite3.connect(DATABASE_NAME, check_same_thread=False)
    conn.row_factory = sqlite3.Row  # Access columns by name
    try:
        yield conn
    finally:
        conn.close()

def init_database():
    """Initialize database and create tables if not exist."""
    with get_connection() as conn:
        cursor = conn.cursor()
        
        # Keywords table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS keywords (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                keyword TEXT UNIQUE NOT NULL,
                is_active BOOLEAN DEFAULT 1,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)
        
        # Articles table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS articles (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                keyword_id INTEGER,
                title TEXT,
                url TEXT,
                publisher TEXT,
                publish_date TIMESTAMP,
                content TEXT,
                sentiment_label TEXT,
                sentiment_score REAL,
                scraped_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY (keyword_id) REFERENCES keywords(id) ON DELETE CASCADE
            )
        """)
        
        # Indexes
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_keyword ON articles(keyword_id)")
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_date ON articles(publish_date)")
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_sentiment ON articles(sentiment_label)")
        
        # Scrape history table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS scrape_history (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                keyword_id INTEGER,
                total_found INTEGER,
                total_success INTEGER,
                positive_count INTEGER,
                negative_count INTEGER,
                neutral_count INTEGER,
                scraped_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY (keyword_id) REFERENCES keywords(id) ON DELETE CASCADE
            )
        """)
        conn.commit()
    
    print("[DB] Database initialized successfully.")

def add_keyword(keyword):
    """Add a new keyword to track."""
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute(
            "INSERT OR IGNORE INTO keywords (keyword) VALUES (?)",
            (keyword,)
        )
        conn.commit()
        
        # Get the keyword ID
        cursor.execute("SELECT id FROM keywords WHERE keyword = ?", (keyword,))
        result = cursor.fetchone()
        return result['id'] if result else None

def get_keywords():
    """Get all active keywords."""
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT id, keyword FROM keywords WHERE is_active = 1 ORDER BY created_at DESC")
        return cursor.fetchall()

def save_article(article, keyword_id):
    """Save a single article to the database. Returns article_id if new, 'duplicate' if exists, None if error."""
    with get_connection() as conn:
        cursor = conn.cursor()
        # Check for duplicate URL
        cursor.execute("SELECT id FROM articles WHERE url = ?", (article.get('url'),))
        if cursor.fetchone():
            return "duplicate"  # Already exists - return marker
        
        # Parse publish date - handle multiple formats
        pub_date = article.get('publish_date')
        if pub_date:
            # SQLite stores dates as strings/timestamps mainly, let's keep it simple or convert to ISO string
            if isinstance(pub_date, datetime):
                pub_date = pub_date.isoformat()
            else:
                try:
                    # Try to normalize string dates to ISO if possible, otherwise store as is
                    from dateutil import parser as date_parser
                    dt = date_parser.parse(str(pub_date))
                    pub_date = dt.isoformat()
                except:
                    pass
        
        cursor.execute("""
            INSERT INTO articles (keyword_id, title, url, publisher, publish_date, content, sentiment_label, sentiment_score)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            keyword_id,
            article.get('title'),
            article.get('url'),
            article.get('publisher'),
            pub_date,
            article.get('text'),
            article.get('sentiment_label'),
            article.get('sentiment_score')
        ))
        conn.commit()
        return cursor.lastrowid

def save_scrape_history(keyword_id, total_found, total_success, pos, neg, neu):
    """Save scrape history for analytics."""
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO scrape_history (keyword_id, total_found, total_success, positive_count, negative_count, neutral_count)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (keyword_id, total_found, total_success, pos, neg, neu))
        conn.commit()

def get_articles(keyword_id=None, date_from=None, date_to=None, limit=100):
    """Get articles with optional filters."""
    query = "SELECT a.*, k.keyword FROM articles a JOIN keywords k ON a.keyword_id = k.id WHERE 1=1"
    params = []
    
    if keyword_id:
        query += " AND a.keyword_id = ?"
        params.append(keyword_id)
    
    if date_from:
        query += " AND a.publish_date >= ?"
        params.append(date_from)
    
    if date_to:
        query += " AND a.publish_date <= ?"
        params.append(date_to)
    
    query += " ORDER BY a.scraped_at DESC LIMIT ?"
    params.append(limit)
    
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute(query, params)
        return cursor.fetchall()

def get_trend_data(keyword_id=None, days=30):
    """Get daily article count and sentiment for trend analysis."""
    # SQLite syntax for date manipulation
    query = """
        SELECT 
            date(scraped_at) as date,
            COUNT(*) as total,
            SUM(CASE WHEN sentiment_label = 'Positive' THEN 1 ELSE 0 END) as positive,
            SUM(CASE WHEN sentiment_label = 'Negative' THEN 1 ELSE 0 END) as negative,
            SUM(CASE WHEN sentiment_label = 'Neutral' THEN 1 ELSE 0 END) as neutral
        FROM articles
        WHERE scraped_at >= date('now', '-' || ? || ' days')
    """
    params = [str(days)]
    
    if keyword_id:
        query += " AND keyword_id = ?"
        params.append(keyword_id)
    
    query += " GROUP BY date(scraped_at) ORDER BY date"
    
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute(query, params)
        return cursor.fetchall()

def get_keyword_comparison():
    """Get article count per keyword for comparison."""
    query = """
        SELECT 
            k.keyword,
            COUNT(a.id) as total_articles,
            SUM(CASE WHEN a.sentiment_label = 'Positive' THEN 1 ELSE 0 END) as positive,
            SUM(CASE WHEN a.sentiment_label = 'Negative' THEN 1 ELSE 0 END) as negative
        FROM keywords k
        LEFT JOIN articles a ON k.id = a.keyword_id
        WHERE k.is_active = 1
        GROUP BY k.id, k.keyword
        ORDER BY total_articles DESC
    """
    
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute(query)
        return cursor.fetchall()

def delete_keyword(keyword_id):
    """Delete a keyword and all its articles."""
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM keywords WHERE id = ?", (keyword_id,))
        conn.commit()

def clear_all_articles():
    """Clear all articles from database (for re-scraping with fixed logic)."""
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM scrape_history")
        cursor.execute("DELETE FROM articles")
        conn.commit()
    print("[DB] All articles cleared. Ready for fresh scrape.")

# Initialize database on import (only creates if not exists)
if __name__ == "__main__":
    init_database()
    print("[DB] Database setup complete.")


In [ ]:
from gnews import GNews
from newspaper import Article
from googlenewsdecoder import new_decoderv1
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import os
import shutil
from datetime import datetime
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

# Initialize IndoBERT Sentiment Model (Lazy Loading)
_sentiment_pipeline = None

def get_sentiment_pipeline():
    """Lazy load the sentiment pipeline to avoid long startup time."""
    global _sentiment_pipeline
    if _sentiment_pipeline is None:
        print("[*] Loading IndoBERT Sentiment Model (first run only)...")
        model_name = "crypter70/IndoBERT-Sentiment-Analysis"
        try:
            _sentiment_pipeline = pipeline(
                "sentiment-analysis",
                model=model_name,
                tokenizer=model_name,
                device=0 if torch.cuda.is_available() else -1
            )
            print("[+] IndoBERT Model loaded successfully.")
        except Exception as e:
            print(f"[!] Log warning: Failed to load sentiment model: {e}")
            return None
    return _sentiment_pipeline

def preprocess_text(text):
    """
    Preprocessing teks sebelum analisis sentimen.
    """
    import re
    
    if not text:
        return ""
    
    # 1. Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 2. Hapus email
    text = re.sub(r'\S+@\S+', '', text)
    
    # 3. Hapus HTML tags (jika ada sisa)
    text = re.sub(r'<[^>]+>', '', text)
    
    # 4. Hapus karakter khusus kecuali tanda baca penting (. , ! ? -)
    text = re.sub(r'[^\w\s.,!?\-]', '', text)
    
    # 5. Hapus whitespace berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 6. Hapus baris kosong berulang
    text = re.sub(r'\n+', '\n', text)
    
    return text

def analyze_sentiment(text):
    """
    Menganalisis sentimen teks menggunakan IndoBERT.
    """
    try:
        # Step 1: Preprocessing - clean the text
        text_clean = preprocess_text(text)
        
        # Step 2: Truncate text to max 512 tokens (BERT limit ~1500 chars)
        text_truncated = text_clean[:1500] if text_clean else ""
        
        if not text_truncated:
            return 0, "Neutral"
        
        # Step 3 & 4: Tokenization + Inference (handled by pipeline)
        pipe = get_sentiment_pipeline()
        if not pipe:
             return 0, "Neutral" # Fallback if model failed to load

        result = pipe(text_truncated)[0]
        
        label = result['label']
        score = result['score']
        
        # Step 5: Normalize labels
        if label.lower() in ['positive', 'positif', 'label_2']:
            return score, "Positive"
        elif label.lower() in ['negative', 'negatif', 'label_0']:
            return score, "Negative"
        else:
            return score, "Neutral"
    except Exception as e:
        print(f"   [!] Sentiment error: {e}")
        return 0, "Neutral"

def decode_google_news_url(google_url):
    """
    Decode URL Google News menggunakan googlenewsdecoder package.
    """
    try:
        # Cek apakah URL adalah Google News URL
        if 'news.google.com' in google_url:
            print(f"   [*] Decoding Google News URL...")
            decoded_url = new_decoderv1(google_url, interval=5)
            
            if decoded_url.get("status"):
                real_url = decoded_url["decoded_url"]
                print(f"   [+] Berhasil decode URL")
                return real_url
            else:
                print(f"   [!] Decoder error: {decoded_url.get('message', 'Unknown error')}")
                return None
        else:
            # Jika bukan Google News URL, kembalikan URL asli
            return google_url
    except Exception as e:
        print(f"   [!] Gagal decode URL: {e}")
        return None

def scrape_google_news(keyword, language='id', country='ID', period='14d', max_results=20):
    """
    Scrape berita dari Google News berdasarkan keyword
    """
    import time as t
    
    # Add small delay to avoid rate limiting
    t.sleep(1)
    
    try:
        google_news = GNews(
            language=language,
            country=country,
            period=period,
            max_results=max_results
        )
        
        news = google_news.get_news(keyword)
        
        if not news:
            print(f"   [!] GNews returned empty for '{keyword}', trying without period filter...")
            # Try without strict period
            google_news_retry = GNews(
                language=language,
                country=country,
                max_results=max_results
            )
            news = google_news_retry.get_news(keyword)
        
        return news if news else []
        
    except Exception as e:
        print(f"   [!] GNews error: {e}")
        return []


def get_article_with_selenium(url):
    """
    Fallback method: Menggunakan Selenium untuk mengambil konten artikel
    yang tidak bisa diambil dengan newspaper3k
    """
    driver = None
    try:
        print(f"   [*] Mencoba dengan Selenium (Kaggle Mode)...")
        
        # Setup Chrome options untuk headless mode di Kaggle
        chrome_options = Options()
        chrome_options.add_argument("--headless")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--window-size=1920,1080")
        chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
        
        # Try to find chrome binary if needed (Kaggle usually has it in path)
        # chrome_options.binary_location = "/usr/bin/google-chrome" # Uncomment if needed on specific envs

        driver = webdriver.Chrome(options=chrome_options)
        driver.get(url)
        
        # Tunggu sampai halaman dimuat
        time.sleep(3)
        
        # Ambil judul
        title = driver.title
        
        # Coba ambil konten dari berbagai selector umum
        content_selectors = [
            "article",
            "[class*='article-content']",
            "[class*='article-body']",
            "[class*='content-body']",
            "[class*='post-content']",
            "[class*='entry-content']",
            ".detail-text",
            ".read__content",
            "#article-content",
            "main"
        ]
        
        content = ""
        for selector in content_selectors:
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                if elements:
                    for el in elements:
                        text = el.text.strip()
                        if len(text) > len(content):
                            content = text
            except:
                continue
        
        # Jika tidak ada konten, ambil body
        if not content:
            try:
                body = driver.find_element(By.TAG_NAME, "body")
                content = body.text[:2000]  # Batasi 2000 karakter
            except:
                pass
        
        if content:
            print(f"   [+] Berhasil ekstrak dengan Selenium")
            return {
                'title': title,
                'text': content,
                'authors': [],
                'publish_date': None,
                'url': url,
                'top_image': None
            }
        
        return None
        
    except Exception as e:
        print(f"   [!] Selenium error: {e}")
        return None
    finally:
        if driver:
            try:
                driver.quit()
            except:
                pass


def get_full_article(url):
    """
    Mendapatkan artikel lengkap dari URL dengan fallback ke Selenium
    """
    # Decode Google News URL jika perlu
    real_url = decode_google_news_url(url)
    
    # Jika decode gagal, return None
    if not real_url:
        print("   [!] Tidak bisa mendapatkan URL asli artikel")
        return None
    
    print(f"   -> URL Asli: {real_url[:60]}...")
    
    # Metode 1: Coba dengan newspaper3k (lebih cepat)
    try:
        print(f"   [*] Mencoba dengan Newspaper3k...")
        article = Article(real_url)
        article.download()
        article.parse()
        
        # Cek apakah berhasil mendapatkan konten
        if article.text and len(article.text) > 100:
            print(f"   [+] Berhasil ekstrak dengan Newspaper3k")
            return {
                'title': article.title,
                'text': article.text,
                'authors': article.authors,
                'publish_date': article.publish_date,
                'url': real_url,
                'top_image': article.top_image
            }
        else:
            print(f"   [!] Newspaper3k: Konten kosong/terlalu pendek")
            
    except Exception as e:
        print(f"   [!] Newspaper3k error: {e}")
    
    # Metode 2: Fallback ke Selenium
    return get_article_with_selenium(real_url)


def save_results(articles, keyword, output_dir="output"):
    """Menyimpan hasil scrape ke CSV dan JSON"""
    import pandas as pd
    
    # Buat folder output jika belum ada
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Generate timestamp untuk nama file yang unik
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    keyword_clean = keyword.replace(" ", "_").lower()
    
    # Convert ke DataFrame
    df = pd.DataFrame(articles)
    
    # Simpan ke CSV
    csv_filename = f"{output_dir}/{keyword_clean}_{timestamp}.csv"
    df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
    print(f"✅ Hasil disimpan ke CSV: {csv_filename}")
    
    # Simpan ke JSON
    json_filename = f"{output_dir}/{keyword_clean}_{timestamp}.json"
    df.to_json(json_filename, orient='records', force_ascii=False, indent=2)
    print(f"✅ Hasil disimpan ke JSON: {json_filename}")
    
    return csv_filename, json_filename


In [ ]:
import time
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

# ==========================================
# ⚙️ CONFIGURATION
# ==========================================
# List of keywords to scrape
KEYWORDS = [
    "PT ASABRI",
    "Korupsi ASABRI",
    "Asuransi Sosial Angkatan Bersenjata"
]

# Scraping settings
SCRAPING_CONFIG = {
    "language": "id",     # 'id' for Indonesia, 'en' for English
    "country": "ID",      # 'ID' for Indonesia, 'US' for USA
    "period": "14d",      # '1h', '1d', '7d', '14d', '1m', '1y' (Time range)
    "max_results": 20,    # Max articles per keyword
    "workers": 5          # Number of parallel threads (higher = faster, but risk of rate limit)
}

# Database settings
RESET_DATABASE = False    # Set to True to wipe all data before scraping
# ==========================================

def process_single_article(news_item, keyword_id, keyword):
    """
    Process a single news item: extract content, analyze sentiment, save to DB.
    """
    try:
        # Get the article title from news_item first to filter early
        news_title = news_item.get('title', '')
        
        # Check if keyword appears in title (case-insensitive)
        keyword_lower = keyword.lower()
        title_lower = news_title.lower()
        
        # Check for main keyword or any significant part
        keyword_parts = [k.strip() for k in keyword_lower.split() if len(k.strip()) > 2]
        keyword_found = any(part in title_lower for part in keyword_parts) or keyword_lower in title_lower
        
        if not keyword_found:
            return ({"title": news_title}, "filtered")
        
        full_article = get_full_article(news_item['url'])
        if full_article:
            # Priority for publish_date
            publish_date = full_article.get('publish_date')
            
            if not publish_date:
                gn_date = news_item.get('published date')
                if gn_date:
                    publish_date = gn_date
            
            full_article['publish_date'] = publish_date
            full_article['publisher'] = news_item.get('publisher', {}).get('title', 'Unknown')
            
            # Sentiment Analysis
            title = full_article.get('title', '')
            content = full_article.get('text', '')
            combined_text = f"{title}. {content}" if content else title
            
            score, label = analyze_sentiment(combined_text)
            full_article['sentiment_score'] = score
            full_article['sentiment_label'] = label
            
            # Save to database
            save_result = save_article(full_article, keyword_id)
            
            if save_result == "duplicate":
                return (full_article, "duplicate")
            elif save_result:
                return (full_article, "new")
            else:
                return (full_article, "failed")
    except Exception as e:
        print(f"[!] Error processing article: {e}")
    return (None, "failed")

def main():
    print("="*60)
    print("   MedMon Headless Scraper (Kaggle Version)")
    print("="*60)
    
    # Initialize Database
    try:
        if RESET_DATABASE:
            try:
                # Close connection if open (handled by context manager usually)
                pass
            except:
                pass
            
            # Simple way to reset: just call clear_all_articles or re-init
            # But db.py doesn't have drop table in init.
            # Let's use clear_all_articles() if tables exist
            init_database() # Ensure tables exist first
            clear_all_articles() # Then clear them
            print("[!] Database CLEARED (RESET_DATABASE=True)")
        else:
            init_database()
            print("[+] Database initialized (Appending to existing data).")
    except Exception as e:
        print(f"[!] Database Error: {e}")
        return

    all_articles = []
    total_keywords = len(KEYWORDS)
    
    for kw_idx, keyword in enumerate(KEYWORDS):
        print(f"\n[{kw_idx+1}/{total_keywords}] Processing keyword: {keyword}")
        
        if kw_idx > 0:
            print("   [*] Waiting 3 seconds to avoid rate limiting...")
            time.sleep(3)
        
        # Get or create keyword in DB
        keyword_id = add_keyword(keyword)
        
        # Search news with retry
        news_results = []
        for attempt in range(2):
            news_results = scrape_google_news(
                keyword=keyword,
                language=SCRAPING_CONFIG["language"],
                country=SCRAPING_CONFIG["country"],
                period=SCRAPING_CONFIG["period"],
                max_results=SCRAPING_CONFIG["max_results"]
            )
            if news_results:
                break
            elif attempt == 0:
                print(f"   [!] No results, retrying with simpler keyword...")
                time.sleep(2)
        
        total_news = len(news_results)
        print(f"   [+] Found {total_news} articles for '{keyword}'")
        
        if total_news == 0:
            print(f"   [!] WARNING: Google News returned 0 results.")
            continue
        
        keyword_articles = []
        pos_count, neg_count, neu_count = 0, 0, 0
        new_count, dup_count, fail_count, filter_count = 0, 0, 0, 0
        
        # Process articles
        with ThreadPoolExecutor(max_workers=SCRAPING_CONFIG["workers"]) as executor:
            future_to_url = {executor.submit(process_single_article, news, keyword_id, keyword): news for news in news_results}
            
            for future in as_completed(future_to_url):
                try:
                    data, status = future.result()
                    
                    if status == "new" and data:
                        new_count += 1
                        data['keyword'] = keyword
                        keyword_articles.append(data)
                        all_articles.append(data)
                        
                        if data['sentiment_label'] == 'Positive':
                            pos_count += 1
                        elif data['sentiment_label'] == 'Negative':
                            neg_count += 1
                        else:
                            neu_count += 1
                            
                        print(f"   [+] NEW: {data['title'][:50]}... ({data['sentiment_label']})")
                    elif status == "duplicate":
                        dup_count += 1
                        print(f"   [=] DUP: {data['title'][:50] if data else 'Unknown'}...")
                    elif status == "filtered":
                        filter_count += 1
                        print(f"   [~] SKIP: No keyword in title")
                    else:
                        fail_count += 1
                        print(f"   [X] FAIL: Extraction failed")
                except Exception as exc:
                    fail_count += 1
                    print(f"   [X] Error: {exc}")
        
        # Print stats
        print(f"\n   📊 Stats for '{keyword}':")
        print(f"      • New articles: {new_count}")
        print(f"      • Duplicates: {dup_count}")
        print(f"      • Filtered: {filter_count}")
        print(f"      • Failed: {fail_count}")
        
        save_scrape_history(keyword_id, total_news, len(keyword_articles), pos_count, neg_count, neu_count)
        
        # Save per-keyword results to JSON/CSV using medmon_kaggle's helper
        if keyword_articles:
            save_results(keyword_articles, keyword, output_dir="output")

    print("\n" + "="*60)
    print(f"🎉 FINISHED! Total new articles: {len(all_articles)}")
    print("="*60)

if __name__ == "__main__":
    main()
